#  Projeto AV1 - Modelagem e Análise de Grafos

Este projeto consiste na construção, análise estatística e visualização de um grafo complexo, focando em métricas de densidade, distribuição de graus e propriedades estruturais de redes sociais.

---

## Contexto e Base de Dados

O tema escolhido para este trabalho foi redes sociais. Para isso, extraímos dados reais do repositório da Universidade de Stanford (SNAP - Stanford Network Analysis Project).

<div align="center">
  <img src="snap base.png" width="600px" alt="Repositórios SNAP">
  <p><i>Figura 1: Visão geral dos datasets de redes sociais disponíveis no repositório SNAP.</i></p>
</div>

### Dataset: Twitch Gamers Social Network
A base de dados específica utilizada foi a **Twitch Gamers**, que mapeia as relações entre usuários da plataforma Twitch. Escolhemos ela pela sua praticidade e.
* **Nós:** 168,114; Representam os usuários (gamers).
* **Arestas:** 6,797,557; Representam amizades ou seguidores mútuos entre esses usuários.
* **Não Direcionado**

---
## Fluxo do Projeto

O pipeline de análise foi dividido nas seguintes etapas:

1.  **Modelagem**: Conversão dos dados brutos em uma estrutura de lista de adjacência.
2.  **Cálculo de Estatísticas**:
    * Número total de vértices (V) e arestas (E).
    * Cálculo da **Densidade do Grafo**.
    * Identificação de **Graus Mínimos, Máximos e Médios**.
3.  **Visualização**:
    * Geração de arquivo `.dot` para representação gráfica.
    * Plotagem da distribuição de graus.
    
    **Formatação** -> **Cálculo** -> **Visualização**

---

## Tecnologias Utilizadas:
* **Library `algs4`**: Biblioteca da Universidade de Princeton para estruturas de dados, mais especificamente a classe `Graph.Java`.
* **Library `matplotli`**: Biblioteca de visualização de dados em Python
* **Library `numpy`**: Biblioteca fundamental para computação numérica em Python
* **Library `powerlaw`**: Biblioteca utilizada para ajustar e analisar distribuições de lei de potência
* **Library `networkx`**:Biblioteca de Python voltada para a criação, manipulação e análise de grafos e redes complexas
* **Library `pydot`**:Biblioteca que permite interagir com arquivos no formato DOT
* **Library `scipy`** :Biblioteca científica para Python utilizada para operações matemáticas e estatísticas avançadas
* **Maven**: Automação de build e gerenciamento de dependências.
* **Graphviz**: Ferramenta utilizada para renderizar as visualizações a partir dos arquivos DOT.
---

# Modelagem
O dataset original da Twitch Gamers foi fornecido em formato CSV contendo pares de usuários conectados.
Para viabilizar a construção do grafo, foi necessário realizar uma etapa de reformatação dos dados para que a biblioteca `algs4` possa ser usada

## `Reformata.py`
Este script tem como objetivo converter o dataset original da Twitch (em CSV) para um formato estruturado que possa ser utilizado na construção do grafo com a biblioteca algs4.

* Transformar pares de usuários conectados em um arquivo de texto contendo:
* Número total de vértices (V)
* Número total de arestas (E)
* Lista de conexões no formato de índices sequenciais

### Etapas do Código
```python
import csv

def csv_to_algs4(input_file, output_file):
    edges = []
    unique_ids = set()
    with open(input_file, newline='') as csvfile:
        reader = csv.DictReader(csvfile)
        for row in reader:
            v = int(row["numeric_id_1"])
            w = int(row["numeric_id_2"])
            edges.append((v, w))
            unique_ids.add(v)
            unique_ids.add(w)

```
Aqui estamos definindo uma função `cvs_to_algs4` que tem como parametros o arquivo de entrada (entrada.cvs) e um arquivo formatado de saida. Depois, é criado `edges` e `unique_ids` que armazenam as arestas e as vertices do grafo. O bloco `with open(input_file,newline='') as cvsfile:`  percorre  todas as linhas do dataset, etraaindo suas ids e atribuindo arestas  aos que estão na mesma linha.

```python
# Mapeia IDs grandes para 0..V-1
    id_to_index = {node_id: idx for idx, node_id in enumerate(sorted(unique_ids))}

    V = len(unique_ids)
    E = len(edges)

    # Escreve no formato algs4
    with open(output_file, 'w') as f:
        f.write(f"{V}\n")
        f.write(f"{E}\n")
        for v, w in edges:
            f.write(f"{id_to_index[v]} {id_to_index[w]}\n")

    print("Conversão concluída!")
    print(f"Vértices: {V}")
    print(f"Arestas: {E}")



if __name__ == "__main__":
    csv_to_algs4("entrada.csv", "grafo_formatado.txt")
```
A segunda parte do `reformata.py` é cria um dicionário (`id_to_index`) que mapeia os IDs originais dos usuários para índices numéricos organizados de `0` até `V-1`(`V` é o número total de vértices). Em seguida, é calculado a quantidade total do `V` e do `E`. Depois, o arquivo de saída é criado no formato exigido pela biblioteca `algs4`, contendo na primeira linha o número de vértices, na segunda o número de arestas e, nas linhas seguintes, cada conexão já convertida para os novos índices sequenciais. Por fim, o bloco `if __name__ == "__main__":` garante que a função seja executada automaticamente quando o script for rodado diretamente, realizando a conversão do arquivo `entrada.csv` para `grafo_formatado.txt` e exibindo no terminal a confirmação com os valores de vértices e arestas processados.

Formatação para `algs4`:

<div align="center">
  <p><i> Vértices: 168114 <br> 
         Arestas: 6797557 <br>
         Vertice  Vertice <br>
         Vertice  Vertice <br>
         Vertice  Vertice <br>
  </i></p>
</div>


---

---
## Algs4 | Graph.Java
O Graph.java é responsável por implementar a estrutura do grafo utilizada no projeto, seguindo o padrão da biblioteca algs4 (de Sedgewick & Wayne).

Ele representa a rede da Twitch como um grafo não-direcionado usando lista de adjacência, que é a forma mais eficiente para grafos grandes e esparsos.Consequentemente foi necessario a implementação de métodos adicionais para a implementação grafica do grafo, que transforma a saida do Graph.Java em um arquivo `.dot`.

### Codigo
```java
public class Graph {
    private final int V;
    private int E;
    private Bag<Integer>[] adj;

public Graph(int V) {
    this.V = V;
    this.E = 0;
    adj = (Bag<Integer>[]) new Bag[V];
    for (int v = 0; v < V; v++) {
        adj[v] = new Bag<Integer>();
    }
}

public Graph(In in) {
        if (in == null) throw new IllegalArgumentException("argument is null");
        try {
            this.V = in.readInt();
            if (V < 0) throw new IllegalArgumentException("number of vertices in a Graph must be non-negative");
            adj = (Bag<Integer>[]) new Bag[V];
            for (int v = 0; v < V; v++) {
                adj[v] = new Bag<Integer>();
            }
            int E = in.readInt();
            if (E < 0) throw new IllegalArgumentException("number of edges in a Graph must be non-negative");
            for (int i = 0; i < E; i++) {
                int v = in.readInt();
                int w = in.readInt();
                validateVertex(v);
                validateVertex(w);
                addEdge(v, w);
            }
        }
        catch (NoSuchElementException e) {
            throw new IllegalArgumentException("invalid input format in Graph constructor", e);
        }
    }


```
A classe Graph representa um grafo não-direcionado implementado por lista de adjacência, onde:

* `V` representa o número total de vértices do grafo
* `E` representa o número total de arestas do grafo
* `adj`é um vetor de listas, onde cada posição do vetor representa um vértice e armazena todos os seus vizinhos

O construtor `Graph` é responsável por criar a estrutura interna do grafo. Ao receber como parâmetro, o número de vértices `V` e inicializar as arestas com 0, ele cria o vetor adj com uma posição para cada vértice, percorrendo todas as posições do vetor e inicializando cada uma com uma nova lista vazia.

A leitura do arquivo `grafo_formatado` ocorre no construtor `Graph(In in)`. Esse construtor utiliza a classe `In` da biblioteca `algs4` interpretando primeiro o número de vértices, depois o número de arestas e, em seguida, cada par de vértices que representa uma conexão. A cada par lido, o método `addEdge` é chamado, construindo progressivamente a estrutura do grafo em memória.


```java
public void addEdge(int v, int w) {
        validateVertex(v);
        validateVertex(w);
        E++;
        adj[v].add(w);
        adj[w].add(v);
    }

    public int degree(int v) {
        validateVertex(v);
        return adj[v].size();
    
```
O método `addEdge(int v, int w)` é responsável por adicionar uma aresta entre dois vértices do grafo, registrado nos dois sentidos. Antes de adicionar a aresta, o método chama `validateVertex(v)` e `validateVertex(w)`, que verificam se os vértices informados estão dentro do intervalo válido (entre 0 e V - 1). Após a validação:

* `E++` incrementa o número total de arestas do grafo.
* adj[v].add(w) adiciona o vértice w na lista de vizinhos de v.
* adj[w].add(v) adiciona o vértice v na lista de vizinhos de w.

Esse processo garante que a estrutura represente corretamente uma conexão bidirecional, como ocorre em redes sociais.

O método `degree(int v)` é responsável por retornar o grau de um vértice. Primeiro é feita a validação com `validateVertex(v)` para garantir que o vértice existe no grafo. Em seguida, o método retorna adj[v].size(), que corresponde à quantidade de elementos armazenados na lista de adjacência daquele vértice.

```java
public void toDotFull(String filename) {
    try (java.io.PrintWriter out = new java.io.PrintWriter(filename)) {
        out.println("graph G {");
        out.println("layout=sfdp;");
        out.println("overlap=false;");
        out.println("node[shape=circle, width=0.25, height=0.25, fontsize=8];");

        for (int v = 0; v < V; v++) {
            for (int w : adj[v]) {
                if (w > v) {
                    out.println(v + " -- " + w + ";");
                }
            }
        }

        out.println("}");
    } catch (java.io.IOException e) {
        e.printStackTrace();
    }
    }

public String toDot() {
        StringBuilder s = new StringBuilder();
        s.append("graph {" + NEWLINE);
       s.append("node[shape=circle, style=filled, fixedsize=true, width=0.3, fontsize=\"10pt\"];" + NEWLINE);


        int selfLoops = 0;

        for (int v = 0; v < V; v++) {
            for (int w : adj[v]) {
                if (v < w) {
                    s.append(v + " -- " + w + NEWLINE);
                }
                else if (v == w) {
                    if (selfLoops % 2 == 0) {
                        s.append(v + " -- " + w + NEWLINE);
                    }
                    selfLoops++;
                }
            }
        }
        s.append("}" + NEWLINE);
        return s.toString();
    }

public static void main(String[] args) {
        if (args.length < 1) {
            System.out.println("Uso: java Graph <caminho_do_arquivo_txt>");
            return;
        }

        // 1. Carrega o grafo do arquivo TXT passado pelo .bat
        In in = new In(args[0]);
        Graph graph = new Graph(in);

        // 2. Lógica de Caminho Portátil
        // Pegamos o caminho do arquivo de entrada (ex: graphs\grafo_formatado.txt)
        java.io.File arquivoEntrada = new java.io.File(args[0]);
        
        // Pegamos a pasta onde esse arquivo está (a pasta "graphs")
        String pastaPai = arquivoEntrada.getParent(); 
        
        // Criamos o novo caminho para o arquivo .dot na mesma pasta
        // Se pastaPai for nulo (arquivo na raiz), usamos apenas o nome
        String caminhoDot = (pastaPai != null) ? 
                            pastaPai + java.io.File.separator + "grafo_full.dot" : 
                            "grafo_full.dot";

        System.out.println("Gerando DOT em: " + caminhoDot);
        
        // 3. Gera o arquivo
        graph.toDotFull(caminhoDot);

        System.out.println("DOT full gerado com sucesso!");
    }

```

Os métodos `toDotFull` e `toDot` são responsáveis por converter o grafo interno em uma representação visual no formato DOT. Ambos percorrem a lista de adjacência do grafo e escrevem cada aresta no padrão exigido pelo Graphviz (v -- w). O método `toDotFull` escreve diretamente em um arquivo `.dot`, permitindo gerar automaticamente um arquivo para visualização, enquanto o método `toDot` constrói e retorna uma String contendo a estrutura completa do grafo em formato DOT, possibilitando manipulação posterior antes da gravação.

O método `main` executa o Graph.java diretamente pelo terminal ou por um arquivo .bat. O funcionamento ocorre em etapas:

* Verifica se foi passado o caminho do arquivo grafo_formatado.txt como argumento.
* Lê o arquivo formatado usando a classe In.
* Constrói o grafo a partir desse arquivo.
* Obtém a pasta onde o arquivo está localizado.
* Cria automaticamente o caminho para salvar o arquivo grafo_full.dot na mesma pasta.
* Chama o método toDotFull() para gerar o arquivo DOT.
* Exibe mensagens de confirmação no console.

A implementação da classe Graph foi fundamental para permitir a extração da distribuição de graus, etapa essencial para a análise estatística da rede e verificação do comportamento de lei de potência.


---
## Dot.py | Geração de Amostra do Grafo
O script dot.py tem como objetivo criar uma versão reduzida do grafo a partir do arquivo grafo_full.dot. O grafo completo da Twitch é muito grande para renderizar no Graphviz, foi necessário gerar uma amostra contendo apenas 1.000 arestas, permitindo a visualização sem comprometer desempenho ou memória do laptop.

```python
import networkx as nx
import pydot
# Caminhos dos seus arquivos
caminho_entrada = "grafo_full.dot"
caminho_saida = "grafo_curto.dot"
print("Lendo o arquivo e pegando apenas os primeiros 1.000 dados (amostra reduceida)...")

try:
    with open(caminho_entrada, 'r', encoding='utf-8') as original:
        with open(caminho_saida, 'w', encoding='utf-8') as novo:
            contador = 0
            for linha in original:
                # Escreve a linha no novo arquivo
                novo.write(linha)
                
                # Se a linha tiver uma conexão (-> ou --), a gente conta
                if '--' in linha or '->' in linha:
                    contador += 1
                
                # Quando chegar em 1.000, para tudo e fecha o grafo
                if contador >= 1000:
                    novo.write("\n}") # Garante que o arquivo feche certinho
                    break
                    
    print(f"Pronto! O arquivo 'grafo_curto.dot' foi criado com {contador} conexões.")
    print("Atenção: esta é apenas uma amostra de 1.000 arestas para viabilizar a execução do Graphviz.")

except Exception as e:
    print(f"Erro: {e}")
```
A entrada do metodo é o `grafo_full.dot` gerado pelo `Graph.java` e gera um arquivo `grafo_curto.dot` na saida. O algoritmo percorre o `grafo_full.dot` linha por linha, copiando cada linha para o novo arquivo,e ao encontrar uma aresta (--), incrementa o contador. Apos o contador chegar em 1000, o programa é interrompido. Assim, é criado uma amostra de 1000 arestas para uma representação grafica menos exigente.



## Cálculo das Estatísticas e Visualização Grafica | Histograma.py
O grafo modelado é não-direcionado, não ponderado e não temporal onde uma aresta é criada apenas quando dois usuários se seguem reciprocamente, caracterizando uma conexão bidirecional. A partir disso, é necessario analisar as suas carateristicas (estrutura, comportamento,densidade) atraves de calculos numericos.

### Distribuição de Grau

Em teoria dos grafos, o **grau de um vértice** representa o número de conexões que esse vértice possui dentro da rede.

Em um grafo não-direcionado, o grau de um vértice (v) é definido como o número de arestas incidentes a ele:

$k_i = \sum_{j} a_{ij}$

* $k_i$ = grau do vértice $i$
* $a_{ij}$ = elemento da matriz de adjacência

No contexto da rede analisada neste projeto:

* Cada **vértice** representa um usuário da plataforma Twitch.
* Cada **aresta** representa uma relação de seguidores mútuos entre dois usuários.

Dessa forma, o grau de um vértice corresponde à **quantidade de conexões mútuas que um usuário possui dentro da rede**.

A partir do cálculo dos graus de todos os vértices do grafo, é possível obter métricas estatísticas importantes que descrevem a estrutura da rede.

#### Codigo
```python
node_index = {}
next_idx = 0
degrees_map = {} # Usaremos um dicionário para contar graus sem criar listas gigantes

print("[INFO] Processando graus do grafo (Streaming)...")

with open(DOT_PATH, 'r', encoding='utf-8') as file:
    for line in file:
        line = line.strip()
        if "--" in line:
            # Parse direto da linha sem guardar na lista
            parts = line.split("--")
            a = parts[0].strip()
            b = parts[1].strip().rstrip(';')
            
            # Contagem de graus direta
            degrees_map[a] = degrees_map.get(a, 0) + 1
            degrees_map[b] = degrees_map.get(b, 0) + 1

# Transforma os valores do dicionário em um array numpy para o restante do código
degrees = np.array(list(degrees_map.values()))
degrees = degrees[degrees > 0] 

print(f"[OK] Graus calculados para {len(degrees):,} nós.")

```
O código apresentado tem como objetivo calcular o grau de cada vértice do grafo a partir do arquivo `.dot` gerado anteriormente. Para isso, foi utilizada uma abordagem de leitura em streaming, na qual o arquivo é processado linha por linha, evitando carregar todo o grafo na memória. Essa estratégia é importante porque a rede analisada possui um grande número de arestas.

Inicialmente, é criado um dicionário (`degrees_map`) responsável por armazenar o grau de cada vértice. A chave do dicionário representa o identificador do usuário (vértice) e o valor representa a quantidade de conexões que ele possui na rede.

Durante a leitura do arquivo `.dot`, o código verifica quais linhas representam arestas do grafo. No formato DOT, uma aresta em grafos não-direcionados é representada pela expressão `v -- w`. Sempre que esse padrão é encontrado, os dois vértices conectados são extraídos da linha.

Para cada aresta identificada, o grau de ambos os vértices é incrementado no dicionário. Isso ocorre porque, em grafos não-direcionados, cada conexão contribui para o grau de ambos os nós envolvidos.

Após o processamento completo do arquivo, os valores armazenados no dicionário são convertidos para um vetor `NumPy`. Essa conversão permite realizar análises estatísticas e gerar gráficos de distribuição de grau posteriormente.

Ao final da execução, o código retorna um vetor contendo o grau de todos os vértices do grafo, que será utilizado para calcular métricas como grau mínimo, grau máximo, grau médio e distribuição de grau da rede.

Quando uma aresta é identificada, a linha é dividida em duas partes para extrair os vértices conectados.

---

### Métricas de Grau Calculadas

Foram calculadas as seguintes métricas:

* **Grau mínimo (k_min)**  
* **Grau máximo (k_max)**  
* **Grau médio (⟨k⟩)**  
* **Mediana do grau**  
* **Desvio padrão**  
* **Coeficiente de Gini**

Essas métricas permitem compreender como as conexões estão distribuídas entre os usuários da rede.

```python
# ================================
# Estatísticas Descritivas
# ================================
def imprimir_estatisticas(degrees):
    print("\n--- Estatísticas da Distribuição de Graus ---")
    print(f"Número total de nós (com grau > 0): {len(degrees)}")
    print(f"Grau Máximo (k_max): {np.max(degrees)}")
    print(f"Grau Mínimo (k_min): {np.min(degrees)}")
    print(f"Grau Médio (<k>): {np.mean(degrees):.2f}")
    print(f"Mediana do Grau: {np.median(degrees)}")
    print(f"Desvio Padrão: {np.std(degrees):.2f}")
    
    # Coeficiente de Gini (Mede desigualdade na conectividade)
    sorted_degrees = np.sort(degrees)
    n = len(degrees)
    index = np.arange(1, n + 1)
    gini = ((np.sum((2 * index - n  - 1) * sorted_degrees)) / (n * np.sum(sorted_degrees)))
    print(f"Coeficiente de Gini: {gini:.4f} (0=igualdade, 1=extrema desigualdade)")

imprimir_estatisticas(degrees)
```
---

### Interpretação dos Resultados

A análise produziu os seguintes valores:

* Número total de nós: **168.114**
* Grau mínimo: **1**
* Grau máximo: **35.279**
* Grau médio: **80.87**
* Mediana do grau: **32**
* Desvio padrão: **314.16**
* Coeficiente de Gini: **0.6809**

#### Grau mínimo

O grau mínimo observado foi **1**, indicando que existem usuários com apenas **uma conexão mútua** dentro da rede. Isso mostra que a rede possui usuários pouco conectados, algo comum em redes sociais reais.

#### Grau máximo

O grau máximo foi **35.279**, indicando a presença de **hubs altamente conectados**. Esses vértices possuem milhares de conexões e exercem grande influência na estrutura da rede.

Em redes sociais, esses hubs normalmente representam usuários populares ou muito ativos na plataforma.

#### Grau médio

$\langle k \rangle = \frac{2E}{N}$

Onde:
* N = número de vértices
* E = número de arestas

O grau médio foi **80.87**, indicando que, em média, cada usuário possui cerca de 81 conexões mútuas.

#### Mediana do grau

A mediana do grau foi **32**, indicando que metade dos usuários possui até **32 conexões mútuas**.

A diferença entre a média (80.87) e a mediana (32) mostra que a distribuição de conexões é **assimétrica**, com poucos vértices concentrando grande parte das conexões.

#### Desvio padrão

$\sigma_k = \sqrt{\frac{1}{N}\sum_{i=1}^{N}(k_i - \langle k \rangle)^2}$
Onde:
* <k> = grau médio.

O desvio padrão foi **314.16**, indicando uma grande variação na quantidade de conexões entre os usuários.

Isso significa que a rede apresenta grande heterogeneidade, com usuários muito pouco conectados coexistindo com hubs extremamente conectados.

#### Coeficiente de Gini

$G = \frac{\sum_{i=1}^{N}\sum_{j=1}^{N} |k_i - k_j|}{2N\sum_{i=1}^{N} k_i}$
Onde:
* $k_i$ = grau do vértice

O coeficiente de Gini encontrado foi **0.6809**.

Esse índice mede o nível de desigualdade na distribuição de conexões:

* 0 → distribuição completamente igual  
* 1 → desigualdade extrema  

O valor obtido indica **alta concentração de conexões em poucos vértices**, reforçando a presença de hubs dominantes na rede.

---
## Densidade
A densidade de um grafo mede a proporção de conexões existentes em relação ao número máximo possível de conexões entre os vértices da rede.Em um grafo não direcionado com N vértices e E arestas, a densidade é definida como:

$D = \frac{2E}{N(N-1)}$

A partir da análise do grafo estudado, foi identificado D = 0.00048104. Esse valor indica que apenas uma pequena fração das conexões possíveis entre os vértices está presente na rede, caracterizando-a como uma rede extremamente esparsa. Esse comportamento é típico de redes sociais e outras redes complexas de grande escala, nas quais cada vértice se conecta apenas a uma pequena parcela do total de vértices existentes.

```python
# ================================
# Cálculo da Densidade
# ================================

N = len(degrees_map)      # número de vértices
sum_degrees = degrees.sum()

E = sum_degrees / 2       # número de arestas

density = (2 * E) / (N * (N - 1))

print("\n--- Métricas da Rede ---")
print(f"Vértices (N): {N}")
print(f"Arestas (E): {int(E)}")
print(f"Densidade: {density:.8f}")
```

## Clustering Médio

O coeficiente de clustering é uma métrica utilizada para medir o nível de agrupamento local em uma rede. Ele quantifica a probabilidade de que dois vértices que são vizinhos de um mesmo vértice também estejam conectados entre si, formando triângulos na rede.
Ele é representado por:
$C = \frac{1}{N}\sum_{i=1}^{N} C_i$

Onde:
* C = coeficiente de clustering médio da rede
* N = número total de vértices
* $C_i$ = coeficiente de clustering do vértice (formula: $C_i = \frac{2T_i}{k_i(k_i - 1)}$)


No grafo analisado, o coeficiente de clustering médio aproximado obtido foi C=0.153. Esse valor indica uma tendência significativa de formação de triângulos na rede, ou seja, existe uma probabilidade relativamente alta de que dois vértices conectados a um mesmo vértice também estejam conectados entre si.

```python
with open("grafo_full.dot", "r", encoding="utf-8") as file:
    for line in file:
        if "--" in line:
            a, b = line.split("--")
            a = a.strip()
            b = b.strip().rstrip(";")

            G.add_edge(a, b)

clustering = nx.approximation.average_clustering(G, trials=1000)

print("Clustering médio aproximado:", clustering)
```

### Distribuição de Grau

Para compreender melhor como os graus estão distribuídos na rede, foi calculada a distribuição de grau.

$P(k) = \frac{N_k}{N}$

Onde:
* $N_k$ = número de vértices com grau $k$
* $N$ = número total de vértices

A distribuição de grau \(P(k)\) representa a probabilidade de um vértice possuir grau \(k\).

Em redes sociais reais, é comum observar que:

* muitos vértices possuem poucos vizinhos
* poucos vértices possuem muitos vizinhos

Esse comportamento gera uma distribuição altamente desigual.

---

### Gráfico de Probabilidade da Distribuição de Grau

O primeiro gráfico gerado foi um **histograma da distribuição de grau**.

Nesse gráfico:

* o eixo **x** representa o grau k
* o eixo **y** representa a frequência ou probabilidade de ocorrência daquele grau

Esse gráfico permite visualizar diretamente como os graus estão distribuídos na rede.

Normalmente, observa-se que a maior parte dos vértices possui **grau baixo**, enquanto poucos vértices possuem graus extremamente elevados.

Esse padrão é típico de **redes sociais complexas**.

---

### Gráfico Log-Log da Distribuição de Grau

Além do gráfico linear, também foi gerado um gráfico em escala log-log. Ela indica qual a chance de escolher aleatoriamente um nó com determinado grau

Nesse tipo de visualização:

* o eixo x é representado em **escala logarítmica**
* o eixo y também é representado em **escala logarítmica**


<div align="center">
  <img src="histogramas.png" width="600px" alt="Repositórios SNAP">
</div>



#### Codigo

```python
# ================================
# Histograma manual (Dados Reais)
# ================================
degree_count = Counter(degrees)
graus = np.array(sorted(degree_count.keys()))
frequencias = np.array([degree_count[g] for g in graus])
pk = frequencias / np.sum(frequencias)

# Filtramos apenas para visualização log-log
log_k = np.log10(graus)
log_pk = np.log10(pk)

```

---

## Lei de Potência na Distribuição de Grau

A lei de potência (power law) é uma distribuição estatística frequentemente observada em redes complexas, como redes sociais e redes da internet. No contexto de grafos, ela descreve como os graus dos nós (número de conexões de cada nó) estão distribuídos na rede.

Essa relação pode ser representada por:

$P(k) \sim k^{-\alpha}$

Onde:
* $k$ = grau do vértice
* $alpha$ = expoente da distribuição

onde P(k) é a probabilidade de um nó ter grau ``k``, e ``alpha`` é o expoente da distribuição. Esse comportamento indica que a maioria dos nós possui poucas conexões, enquanto poucos nós possuem muitas conexões. Esses nós altamente conectados são chamados de hubs.

Quando uma rede apresenta esse tipo de distribuição, ela é chamada de rede ``scale-free``, caracterizada por uma estrutura heterogênea, na qual alguns poucos nós concentram grande parte das conexões da rede.

Nesse processo, o vetor degrees contém o grau de cada nó da rede. A função powerlaw.Fit() ajusta um modelo de lei de potência aos dados e estima o valor do expoente α.

O valor do expoente pode ser calculado usando:
$\alpha = 1 + n \left[\sum_{i=1}^{n} \ln\left(\frac{k_i}{x_{\min}}\right)\right]^{-1}$

Onde:
* $k_i$ = graus observados
* $x_min$ = valor mínimo considerado
* $n$ = número de observações

No grafo analisado, foi obtido um valor de α ≈ 2.54, que está dentro da faixa típica observada em redes sociais reais (2 < α < 3).

---
## Análise da Lei de Potência no Grafo

Para verificar se o grafo analisado segue uma lei de potência, foi utilizado o seguinte código em Python com a biblioteca `powerlaw`:

```python
import powerlaw

fit = powerlaw.Fit(degrees)

alpha = fit.alpha
xmin = fit.xmin

print(f"Expoente estimado (alpha): {alpha:.3f}")
print(f"Grau mínimo considerado (xmin): {xmin}")
````
---

### Interpretação Estrutural da Rede

A combinação das métricas e da análise da distribuição de grau indica que:

* existem **hubs extremamente conectados**
* a rede apresenta **alta desigualdade de conexões**
* poucos vértices concentram grande parte das arestas
* a distribuição segue aproximadamente uma **lei de potência**

Essas características são típicas de redes sociais online, nas quais a popularidade tende a se concentrar em um pequeno número de usuários altamente conectados.

### Comparação de Modelos Estatísticos da Distribuição de Grau

$R = \sum_{i=1}^{n} \left[\ln p_1(x_i) - \ln p_2(x_i)\right]$
Onde:
* $p1$ = distribuição power law
* $p2$ = distribuição exponencial


Após ajustar a distribuição de graus do grafo a uma lei de potência, é importante verificar se esse modelo realmente descreve bem os dados. Para isso, realizamos uma comparação estatística entre dois modelos de distribuição:

- **Power Law (Lei de Potência)**
- **Distribuição Exponencial**

Essa comparação permite avaliar qual modelo explica melhor o comportamento observado na rede.

---

#### Código Utilizado

```python
# ================================
# Comparação de Modelos (Relação de Verossimilhança)
# ================================
R, p = fit.distribution_compare('power_law', 'exponential', normalized_ratio=True)

print(f"\n--- Comparação de Modelos ---")
print(f"Power Law vs. Exponencial: R={R:.2f}, p-value={p:.4f}")

if R > 0 and p < 0.05:
    print("Resultado: A Lei de Potência é o melhor ajuste.")
else:
    print("Resultado: Não há evidência estatística de que seja Lei de Potência pura.")

# Interpretação do Alpha
def interpretar_alpha(a):
    print(f"\n--- Interpretação do Expoente α ({a:.2f}) ---")
    
    if 2 < a < 3:
        print("Regime 'Scale-Free': O grafo é ultra-pequeno. Hubs dominam a topologia.")
        
    elif a <= 2:
        print("Regime de Hubs Anômalos: O maior nó cresce com o tamanho da rede.")
        
    else:
        print("Decaimento Rápido: A rede se comporta de forma mais similar a uma rede aleatória.")

interpretar_alpha(alpha)
```
<div align="center">
  <img src="analise_texto_resumo.png" width="600px" alt="">
</div>

## Análise de Topologia: Visualização de Amostra em PDF
A visualização foi gerada utilizando a ferramenta Graphviz a partir de um arquivo no formato DOT. Devido ao grande tamanho da rede completa, foi utilizada uma amostra de 1000 nós, permitindo observar a estrutura básica do grafo e alguns padrões de conectividade.

Para compreender a estrutura de conexões da rede social da Twitch, foi gerada uma visualização utilizando a ferramenta Graphviz a partir de um arquivo no formato DOT. Devido ao grande tamanho da rede completa, foi utilizada uma amostra de 1000 nós, permitindo observar a estrutura básica do grafo e alguns padrões de conectividade. 

Na visualização gerada, podemos observar os seguintes elementos:

* Nós (Nodes): representam os usuários ou streamers presentes na plataforma.

* Arestas (Edges): indicam as conexões entre esses usuários, como relações de seguimento ou interação dentro da rede.

Densidade da Rede: mesmo considerando apenas uma pequena amostra, é possível perceber que nem todos os nós estão conectados entre si, refletindo o comportamento típico de redes sociais reais. No grafo completo analisado, a densidade observada foi aproximadamente 0,00048, indicando que a rede é esparsa, ou seja, apenas uma pequena fração das conexões possíveis entre os usuários realmente existe.

Além disso, a visualização da amostra permite observar a formação de pequenos agrupamentos de nós (clusters) e a presença de vértices com maior número de conexões. Esse comportamento sugere características comuns de redes sociais, nas quais alguns usuários possuem maior centralidade dentro da estrutura da rede.